<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Auditoría de una Encuesta Cafetera Ficticia ☕
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 00
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/00_Fundamentos_Estadistica_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 00: Fundamentos de Estadística y Análisis de Datos** ([README del módulo](../00%20-%20Fundamentos%20de%20Estadistica%20y%20Analisis%20de%20Datos/README.md)) con un problema **nuevo**: la auditoría de las encuestas de una **cooperativa cafetera ficticia** de 6 000 fincas repartidas en tres estratos (valle, ladera y páramo).

Al terminar serás capaz de:

1. **Clasificar variables** por su escala de medición y **diseñar una muestra**: comparar el muestreo aleatorio simple con el estratificado proporcional y con la **asignación de Neyman**, primero con la **varianza teórica** y luego con una **simulación**.
2. **Separar sesgo y varianza** con la descomposición $\text{ECM}=\text{varianza}+\text{sesgo}^2$, comprobar que **más datos no corrigen el sesgo de no respuesta** y corregirlo parcialmente con **ponderación por el inverso de la probabilidad de respuesta**.
3. **Distinguir correlación de causalidad**: calcular Pearson y Spearman, la **correlación parcial** y la pendiente **ajustada** por un confusor, y contrastarlas con un **experimento aleatorizado**.
4. **Reconocer la paradoja de Simpson** y **estandarizar** tasas, y medir con una simulación qué tanto crece el riesgo de falsos descubrimientos al mirar muchas métricas (**comparaciones múltiples**).

> 📚 **Repasa si lo necesitas:** [Introducción y resumen del curso](../00%20-%20Fundamentos%20de%20Estadistica%20y%20Analisis%20de%20Datos/00_Introduccion_y_Resumen_del_Curso.ipynb) · [Conceptos estadísticos en ciencia de datos](../00%20-%20Fundamentos%20de%20Estadistica%20y%20Analisis%20de%20Datos/01_Conceptos_Estadisticos_en_Ciencia_de_Datos.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija; la cooperativa y sus cifras son **inventadas**). Bibliotecas: `numpy`, `pandas` y `scipy`. Todo el cuaderno corre en pocos segundos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import math

import numpy as np
import pandas as pd
from scipy import stats

SEMILLA = 2025


def generar_poblacion(semilla=SEMILLA):
    """Censo ficticio de 6 000 fincas cafeteras en tres estratos.

    sacos: producción anual (sacos) | satisfaccion: nota de 0 a 10 con la cooperativa |
    p_resp: probabilidad (real, oculta en la práctica) de que la finca conteste una encuesta |
    quintil: quintil de `sacos` (dato administrativo, conocido para todas las fincas).
    """
    rng = np.random.default_rng(semilla)
    partes = []
    for estrato, N_h, (m, s) in [("valle", 3000, (40.0, 4.0)), ("ladera", 2400, (25.0, 14.0)), ("paramo", 600, (12.0, 20.0))]:
        if estrato == "valle":
            sacos = rng.normal(m, s, N_h)
        else:                                      # gamma con media m y desviación s (asimétrica)
            sacos = rng.gamma((m / s) ** 2, s ** 2 / m, N_h)
        partes.append(pd.DataFrame({"estrato": estrato, "sacos": np.clip(sacos, 0.5, None).round(1)}))
    df = pd.concat(partes, ignore_index=True)
    n = len(df)
    df.insert(0, "finca_id", [f"F-{i:04d}" for i in range(1, n + 1)])
    df["categoria"] = pd.qcut(df["sacos"].rank(method="first"), 3, labels=["baja", "media", "alta"]).astype(str)
    df["anio_fundacion"] = rng.integers(1950, 2021, n)
    df["n_trabajadores"] = 1 + rng.poisson(1.5 + df["sacos"] / 10)
    df["satisfaccion"] = np.clip(3 + 0.1 * df["sacos"] + rng.normal(0, 1.5, n), 0, 10).round(2)
    df["p_resp"] = 1 / (1 + np.exp(-(-1.3 + 0.06 * (df["sacos"] - 30))))
    df["quintil"] = pd.qcut(df["sacos"].rank(method="first"), 5, labels=False)
    return df


def generar_parcelas(semilla=SEMILLA, n=1500):
    """Parcelas con fertilidad (Z), horas de asistencia técnica (T) y rendimiento (Y). Efecto causal real de T sobre Y: 0.5."""
    rng = np.random.default_rng(semilla + 10)
    z = rng.normal(0, 1, n)
    t = 5 + 2.0 * z + rng.normal(0, 1.5, n)            # las parcelas fértiles reciben más asistencia
    y = 20 + 3.0 * z + 0.5 * t + rng.normal(0, 2, n)
    return pd.DataFrame({"fertilidad": z, "horas": t, "rendimiento": y})


def generar_experimento(semilla=SEMILLA, n=600):
    """Experimento: las horas de asistencia se asignan AL AZAR (independientes de la fertilidad)."""
    rng = np.random.default_rng(semilla + 11)
    z = rng.normal(0, 1, n)
    t = rng.uniform(0, 10, n)
    y = 20 + 3.0 * z + 0.5 * t + rng.normal(0, 2, n)
    return pd.DataFrame({"fertilidad": z, "horas": t, "rendimiento": y})


poblacion = generar_poblacion()
print(f"Censo ficticio: {len(poblacion)} fincas | estratos: {poblacion['estrato'].value_counts().to_dict()}")
print(poblacion.head(3).to_string(index=False))

---
## Reto 1: ¿Cómo muestrear a la cooperativa? 🎯

**Contexto:** quieres estimar la **producción media por finca** (`sacos`) de la cooperativa con una muestra de `n = 150` fincas. Tienes el censo completo (`poblacion`), así que conoces el **parámetro verdadero** y puedes medir qué tan bueno es cada diseño: **MAS** (muestreo aleatorio simple sin reemplazo), **estratificado proporcional** y **estratificado con asignación de Neyman** ($n_h \propto N_h S_h$).

**Enunciado**
1. `escalas`: clasifica cada columna como `'nominal'`, `'ordinal'`, `'intervalo'` o `'razon'` (sin tilde).
2. `mu`: media poblacional de `sacos`; `S2`: varianza poblacional con divisor $N-1$ (la $S^2$ de la teoría de muestreo); `N_h` y `S_h`: tamaño y desviación estándar (divisor $N_h-1$) de cada estrato (Series indexadas por estrato).
3. `n_prop` y `n_ney`: asignaciones **enteras** (usa `np.round`) de las `n` fincas a los estratos: $n_h = n\,N_h/N$ y $n_h = n\,N_h S_h / \sum_k N_k S_k$.
4. Varianzas **teóricas** de la media muestral: `var_mas(N, S2, n)` $=\left(1-\frac{n}{N}\right)\frac{S^2}{n}$ y `var_estratificado(N_h, S_h, n_h)` $=\sum_h W_h^2\left(1-\frac{n_h}{N_h}\right)\frac{S_h^2}{n_h}$ con $W_h=N_h/N$. Guarda `v_teo` (diccionario con las llaves `'mas'`, `'prop'`, `'ney'`).
5. Simula `R = 2000` encuestas por diseño con `estimar(diseno, rng)` (el estimador estratificado es $\sum_h W_h \bar y_h$). Guarda las estimaciones en `est` (diccionario de arreglos), sus varianzas empíricas en `v_emp` (`ddof=1`) y el **efecto de diseño** `deff[d] = v_emp[d] / v_emp['mas']`.

**Criterios de aceptación:** los parámetros coinciden con un cálculo independiente; `v_teo['ney'] < v_teo['prop'] < v_teo['mas']`; cada varianza empírica queda a menos de 15 % de la teórica y cada estimador es insesgado dentro del error de la simulación.

**Puntaje sugerido:** 20 puntos.

In [ ]:
n = 150      # tamaño de muestra de los tres diseños
R = 2000     # réplicas de la simulación

# TODO: completa
escalas = {"estrato": None, "categoria": None, "anio_fundacion": None, "sacos": None, "n_trabajadores": None}

N = len(poblacion)
mu = None          # media poblacional de `sacos`
S2 = None          # varianza poblacional (divisor N - 1)
N_h = None         # Serie: tamaño de cada estrato
S_h = None         # Serie: desviación estándar (divisor N_h - 1) de sacos en cada estrato
n_prop = None      # Serie: asignación proporcional (enteros)
n_ney = None       # Serie: asignación de Neyman (enteros)


def var_mas(N, S2, n):
    # TODO: varianza teórica de la media muestral bajo MAS sin reemplazo
    return None


def var_estratificado(N_h, S_h, n_h):
    # TODO: varianza teórica del estimador estratificado  Σ W_h² (1 - n_h/N_h) S_h² / n_h
    return None


v_teo = None       # {'mas': ..., 'prop': ..., 'ney': ...}


def estimar(diseno, rng):
    # TODO: extrae UNA muestra con el diseño 'mas' | 'prop' | 'ney' y devuelve la estimación de la media poblacional
    return None


rng = np.random.default_rng(SEMILLA + 1)
est = None         # {diseno: arreglo con R estimaciones}
v_emp = None       # {diseno: varianza empírica (ddof=1) de las R estimaciones}
deff = None        # {diseno: v_emp[diseno] / v_emp['mas']}

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert escalas == {"estrato": "nominal", "categoria": "ordinal", "anio_fundacion": "intervalo", "sacos": "razon", "n_trabajadores": "razon"}, "escalas: estrato = nominal (solo etiqueta), categoria = ordinal (hay orden pero no distancias), anio_fundacion = intervalo (el año 0 no significa «sin año»), sacos y n_trabajadores = razon (el cero es ausencia real)."
_y = poblacion["sacos"].to_numpy(float)
_g = poblacion.groupby("estrato")["sacos"]
_idx = sorted(poblacion["estrato"].unique())
assert mu is not None and abs(mu - _y.mean()) < 1e-9 and abs(S2 - _y.var(ddof=1)) < 1e-9, "mu es la media de todo el censo y S2 su varianza con divisor N - 1 (ddof=1)."
assert N_h is not None and S_h is not None and [int(N_h[h]) for h in _idx] == [int(_g.size()[h]) for h in _idx] and np.allclose([S_h[h] for h in _idx], [_g.std(ddof=1)[h] for h in _idx]), "N_h y S_h: usa groupby('estrato') y, para S_h, std(ddof=1)."
_NH = np.array([N_h[h] for h in _idx], float); _SH = np.array([S_h[h] for h in _idx], float)
_ip = n * _NH / N; _in = n * _NH * _SH / (_NH * _SH).sum()
assert n_prop is not None and np.all(np.abs(np.array([n_prop[h] for h in _idx]) - _ip) <= 0.5 + 1e-9) and abs(sum(n_prop) - n) <= 1, "n_prop: n * N_h / N redondeado a enteros (la suma puede diferir de n en 1 por el redondeo)."
assert n_ney is not None and np.all(np.abs(np.array([n_ney[h] for h in _idx]) - _in) <= 0.5 + 1e-9) and abs(sum(n_ney) - n) <= 1, "n_ney: n * N_h * S_h / suma(N_k * S_k) redondeado a enteros."
assert abs(var_mas(N, S2, n) - (1 - n / N) * S2 / n) < 1e-12, "var_mas = (1 - n/N) * S2 / n  (la corrección de población finita 1 - n/N sí importa aquí)."
_f = lambda nh: float(np.sum((_NH / N) ** 2 * (1 - nh / _NH) * _SH ** 2 / nh))
assert abs(var_estratificado(N_h, S_h, n_prop) - _f(np.array([n_prop[h] for h in _idx], float))) < 1e-9, "var_estratificado = suma de W_h² (1 - n_h/N_h) S_h² / n_h sobre los estratos."
assert isinstance(v_teo, dict) and set(v_teo) == {"mas", "prop", "ney"} and v_teo["ney"] < v_teo["prop"] < v_teo["mas"], "Debe cumplirse v_teo['ney'] < v_teo['prop'] < v_teo['mas']: estratificar ayuda y repartir según N_h·S_h ayuda más."
assert abs(v_teo["ney"] - _f(np.array([n_ney[h] for h in _idx], float))) < 1e-9, "v_teo['ney'] se calcula con la asignación de Neyman n_ney."
for _d in ("mas", "prop", "ney"):
    assert len(est[_d]) == R, f"est['{_d}'] debe tener R = {R} estimaciones."
    assert abs(v_emp[_d] / v_teo[_d] - 1) < 0.15, f"La varianza empírica de '{_d}' debe quedar a menos de 15 % de la teórica; revisa el estimador (¿pesos W_h = N_h/N? ¿muestreo sin reemplazo?)."
    assert abs(est[_d].mean() - mu) < 4 * math.sqrt(v_teo[_d] / R), f"El estimador '{_d}' debería ser insesgado: su promedio sobre las réplicas debe estar cerca de mu."
assert deff["mas"] == 1 and deff["ney"] < deff["prop"] < 1, "El efecto de diseño de MAS es 1; los estratificados deben quedar por debajo de 1 y Neyman por debajo del proporcional."
print(f"✅ Reto 1 superado: media = {mu:.2f} sacos | varianza teórica MAS {v_teo['mas']:.3f}, proporcional {v_teo['prop']:.3f}, Neyman {v_teo['ney']:.3f} | deff simulado: prop {deff['prop']:.2f}, Neyman {deff['ney']:.2f}")

---
## Reto 2: La encuesta de satisfacción que solo contestan los grandes 📉

**Contexto:** la cooperativa envía una encuesta de **satisfacción** (`satisfaccion`, de 0 a 10) a `n` fincas elegidas al azar, pero **las fincas más productivas contestan más** (su probabilidad real de responder es `p_resp`, que crece con `sacos`). Como además la satisfacción sube con la producción, el promedio de quienes responden **sobreestima** la satisfacción de la cooperativa. `quintil` (quintil de `sacos`) es conocido para **todas** las fincas.

**Enunciado**
1. `mu_y`: satisfacción media verdadera. `sesgo_teorico`: sesgo aproximado del promedio simple de quienes responden, $\dfrac{\sum_i p_i y_i}{\sum_i p_i}-\mu_y$.
2. `una_encuesta(rng, n)` → `(ingenuo, ipw, post)`: invita a `n` fincas al azar **sin reemplazo**; cada una responde con probabilidad `p_resp`. Estimadores: **ingenuo** (promedio simple de quienes responden), **ipw** (promedio ponderado con pesos $1/p_i$, dividido por la suma de pesos) y **post** (pesos $1/\hat r_q$, con $\hat r_q$ la tasa de respuesta **observada en la muestra** dentro de cada quintil; no usa `p_resp`).
3. `sim`: `DataFrame` con las columnas `ingenuo`, `ipw`, `post` tras `R2 = 1000` encuestas de `n = 400`. `tabla`: `DataFrame` indexado por esas tres columnas con `sesgo`, `varianza` (con `ddof=0`) y `ecm` (error cuadrático medio respecto de `mu_y`).
4. `tam`: diccionario `{n: (sesgo, varianza)}` del estimador **ingenuo** para `n = 100` (400 réplicas) y `n = 1600` (400 réplicas).

**Criterios de aceptación:** `ecm = varianza + sesgo²` en cada estimador; el sesgo simulado del ingenuo coincide con `sesgo_teorico` dentro del error de la simulación; `ipw` y `post` reducen el sesgo en más de la mitad; y al multiplicar `n` por 16 la varianza baja a una fracción pequeña pero el **sesgo casi no cambia**.

**Puntaje sugerido:** 25 puntos.

In [ ]:
y = poblacion["satisfaccion"].to_numpy()
p = poblacion["p_resp"].to_numpy()
q = poblacion["quintil"].to_numpy()
R2 = 1000

# TODO: completa
mu_y = None            # satisfacción media verdadera
sesgo_teorico = None   # sum(p*y)/sum(p) - mu_y


def una_encuesta(rng, n):
    # TODO: invita a n fincas al azar, simula quién responde y devuelve (ingenuo, ipw, post)
    return None, None, None


rng = np.random.default_rng(SEMILLA + 2)
sim = None             # DataFrame con R2 filas y columnas ingenuo, ipw, post (n = 400)
tabla = None           # índice: ingenuo, ipw, post | columnas: sesgo, varianza, ecm
tam = None             # {100: (sesgo, varianza), 1600: (sesgo, varianza)} del estimador ingenuo

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert mu_y is not None and abs(mu_y - y.mean()) < 1e-9, "mu_y es el promedio de `satisfaccion` de TODAS las fincas."
assert abs(sesgo_teorico - ((p * y).sum() / p.sum() - y.mean())) < 1e-9 and sesgo_teorico > 0.2, "sesgo_teorico = sum(p*y)/sum(p) - mu_y: debe ser positivo (responden más las fincas satisfechas)."
assert sim is not None and list(sim.columns) == ["ingenuo", "ipw", "post"] and len(sim) == R2, "sim debe tener R2 filas y las columnas ingenuo, ipw, post."
assert sim.notna().all().all(), "Alguna estimación quedó vacía: ¿dividiste por la suma de pesos y usaste solo a quienes responden?"
assert list(tabla.index) == ["ingenuo", "ipw", "post"] and list(tabla.columns) == ["sesgo", "varianza", "ecm"], "tabla: índice ingenuo/ipw/post y columnas sesgo/varianza/ecm."
assert np.allclose(tabla["sesgo"], sim.mean() - y.mean()) and np.allclose(tabla["varianza"], sim.var(ddof=0)), "sesgo = promedio de las estimaciones - mu_y; varianza = var(ddof=0) de las estimaciones."
assert np.allclose(tabla["ecm"], ((sim - y.mean()) ** 2).mean()) and np.allclose(tabla["ecm"], tabla["varianza"] + tabla["sesgo"] ** 2), "ecm = promedio de (estimación - mu_y)² y debe cumplir ecm = varianza + sesgo²."
_se = sim["ingenuo"].std() / math.sqrt(R2)
assert abs(tabla.loc["ingenuo", "sesgo"] - sesgo_teorico) < 5 * _se + 0.03, "El sesgo simulado del promedio ingenuo debe parecerse al sesgo teórico (¿simulaste la respuesta con rng.random(n) < p[idx]?)."
_b = tabla["sesgo"].abs()
assert _b["ipw"] < 0.35 * _b["ingenuo"] and _b["post"] < 0.6 * _b["ingenuo"], "Las ponderaciones deben reducir el sesgo: ipw a menos de 35 % del ingenuo y post a menos de 60 %."
assert set(tam) == {100, 1600} and tam[1600][1] < tam[100][1] / 6, "Con n 16 veces mayor la varianza del ingenuo debe bajar a menos de 1/6 de la de n = 100."
assert 0.6 < tam[1600][0] / tam[100][0] < 1.6, "El sesgo del ingenuo NO debe cambiar mucho al aumentar n: más datos reducen la varianza, no el sesgo."
print(f"✅ Reto 2 superado: sesgo ingenuo {tabla.loc['ingenuo','sesgo']:+.3f} (teórico {sesgo_teorico:+.3f}) | ipw {tabla.loc['ipw','sesgo']:+.3f} | post {tabla.loc['post','sesgo']:+.3f} | n=100→1600: varianza {tam[100][1]:.4f}→{tam[1600][1]:.4f}, sesgo {tam[100][0]:+.3f}→{tam[1600][0]:+.3f}")

---
## Reto 3: ¿La asistencia técnica sube el rendimiento? 🔗

**Contexto:** `parcelas` registra la **fertilidad** del suelo, las **horas de asistencia técnica** que recibió cada parcela y su **rendimiento**. Las parcelas fértiles reciben más asistencia, así que la asociación observada mezcla el efecto de la asistencia con el de la fertilidad (un **confusor**). Además hay un `experimento` en el que las horas se asignaron **al azar**. Por construcción, el efecto causal real de una hora más es **0.5**, pero tú no lo sabes: debes estimarlo.

**Enunciado**
1. `r_pearson` y `r_spearman` entre `horas` y `rendimiento` (Pearson **desde cero** con covarianza y desviaciones; Spearman como Pearson de los **rangos**, con `stats.rankdata`).
2. `r_parcial`: correlación parcial entre `horas` y `rendimiento` **controlando** `fertilidad`, por el método de los **residuos** (restas la recta de cada variable contra la fertilidad y correlacionas los residuos). `r_parcial_f`: lo mismo con la fórmula $\dfrac{r_{TY}-r_{TZ}\,r_{YZ}}{\sqrt{(1-r_{TZ}^2)(1-r_{YZ}^2)}}$.
3. `pend_ingenua`: pendiente de la regresión simple de `rendimiento` sobre `horas`; `pend_ajustada`: coeficiente de `horas` en la regresión de `rendimiento` sobre `horas` y `fertilidad` (mínimos cuadrados con `np.linalg.lstsq`); `pend_experimento`: pendiente simple en `experimento`.

**Criterios de aceptación:** tus coeficientes coinciden con un cálculo independiente; los dos cálculos de la correlación parcial coinciden; la pendiente ingenua es **más del doble** del efecto real, y la ajustada y la del experimento caen cerca de él.

**Puntaje sugerido:** 20 puntos.

In [ ]:
parcelas = generar_parcelas()
experimento = generar_experimento()

# TODO: completa
r_pearson = None      # Pearson entre horas y rendimiento, desde cero
r_spearman = None     # Pearson de los rangos
r_parcial = None      # correlación parcial por residuos
r_parcial_f = None    # correlación parcial por la fórmula
pend_ingenua = None   # pendiente de rendimiento ~ horas
pend_ajustada = None  # coeficiente de horas en rendimiento ~ horas + fertilidad
pend_experimento = None  # pendiente de rendimiento ~ horas en el experimento aleatorizado

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_t, _y, _z = (parcelas[c].to_numpy() for c in ("horas", "rendimiento", "fertilidad"))
assert r_pearson is not None and abs(r_pearson - np.corrcoef(_t, _y)[0, 1]) < 1e-9, "r_pearson = cov(T, Y) / (sd(T) · sd(Y)); compara con np.corrcoef."
assert abs(r_spearman - stats.spearmanr(_t, _y)[0]) < 1e-9, "r_spearman = Pearson de los rangos (stats.rankdata de cada variable)."
_e = (np.corrcoef(_t, _y)[0, 1] - np.corrcoef(_t, _z)[0, 1] * np.corrcoef(_y, _z)[0, 1]) / math.sqrt((1 - np.corrcoef(_t, _z)[0, 1] ** 2) * (1 - np.corrcoef(_y, _z)[0, 1] ** 2))
assert r_parcial is not None and r_parcial_f is not None and abs(r_parcial_f - _e) < 1e-9 and abs(r_parcial - _e) < 1e-6, "Los residuos de ajustar cada variable contra la fertilidad (np.polyfit) deben dar la misma correlación que la fórmula."
assert abs(r_parcial) < abs(r_pearson) / 2, "Al controlar la fertilidad la correlación debe bajar a menos de la mitad: parte de la asociación era del confusor."
_X = np.column_stack([np.ones_like(_t), _t, _z])
assert abs(pend_ingenua - np.polyfit(_t, _y, 1)[0]) < 1e-9, "pend_ingenua = pendiente de la recta rendimiento ~ horas (np.polyfit(horas, rendimiento, 1)[0])."
assert abs(pend_ajustada - np.linalg.lstsq(_X, _y, rcond=None)[0][1]) < 1e-9, "pend_ajustada: columnas [1, horas, fertilidad] en lstsq; toma el coeficiente de horas."
assert abs(pend_experimento - np.polyfit(experimento["horas"], experimento["rendimiento"], 1)[0]) < 1e-9, "pend_experimento: la misma pendiente simple, pero con los datos de `experimento`."
assert pend_ingenua > 2 * 0.5, "La pendiente ingenua debería sobreestimar el efecto real (0.5) por más del doble: es la mezcla de asistencia y fertilidad."
assert abs(pend_ajustada - 0.5) < 0.15 and abs(pend_experimento - 0.5) < 0.2, "Ajustando por la fertilidad, o asignando al azar, la pendiente debe quedar cerca del efecto real 0.5."
print(f"✅ Reto 3 superado: Pearson {r_pearson:.3f}, Spearman {r_spearman:.3f}, parcial {r_parcial:.3f} | pendiente ingenua {pend_ingenua:.2f}, ajustada {pend_ajustada:.2f}, experimento {pend_experimento:.2f} (real 0.5)")

---
## Reto 4: Dos paradojas del agregado y del azar 🔄

**Contexto (a):** dos cooperativas, **A** y **B**, reportan cuántos lotes de café pasaron la **inspección de calidad** según su dificultad (`facil` o `dificil`; las cifras son inventadas). B parece mejor en el total, pero ¿lo es? **Contexto (b):** un equipo revisa **20 métricas** de una plataforma tras un cambio que, en realidad, no hizo nada; quieres saber con qué frecuencia al menos una métrica sale «significativa» solo por azar.

**Enunciado**
1. `tasas`: `DataFrame` indexado por `coop` (`A`, `B`) con las columnas `facil`, `dificil` y `global` (proporción de lotes aprobados). `simpson`: `True` si una cooperativa gana en **cada** categoría y pierde en el **total**.
2. `tasa_est`: tasas **estandarizadas** (Series indexada por `coop`) aplicando a ambas cooperativas **los mismos pesos**: la mezcla de lotes de las dos juntas (la proporción de lotes fáciles y difíciles sobre `lotes` total).
3. `fwer_teorico(m, alfa)`: probabilidad de al menos un falso positivo con $m$ pruebas independientes, $1-(1-\alpha)^m$.
4. Con `R4 = 2000` repeticiones de `m = 20` comparaciones de dos grupos de 30 (todo normal, **sin efecto real**; usa `stats.ttest_ind` con `axis`), guarda `fwer_sim` (fracción de repeticiones con al menos un `p < 0.05`) y `fwer_bonf` (igual, pero con umbral `0.05 / m`).

**Criterios de aceptación:** las tasas coinciden con un cálculo independiente; `simpson` es `True`; con pesos comunes la cooperativa que perdía en el total pasa a ganar; y el FWER simulado concuerda con el teórico (≈ 0.64), mientras que el de Bonferroni se acerca a 0.05.

**Puntaje sugerido:** 20 puntos.

In [ ]:
lotes = pd.DataFrame({"coop": ["A", "A", "B", "B"], "dificultad": ["facil", "dificil", "facil", "dificil"],
                      "lotes": [30, 270, 270, 30], "aprobados": [28, 162, 243, 15]})
R4, m = 2000, 20
rng = np.random.default_rng(SEMILLA + 3)

# TODO: completa
tasas = None        # DataFrame: índice coop; columnas facil, dificil, global
simpson = None      # True si una coop gana en cada categoría pero pierde en el total
tasa_est = None     # Serie: tasa estandarizada con la mezcla común de lotes


def fwer_teorico(m, alfa):
    # TODO: probabilidad de al menos un falso positivo con m pruebas independientes
    return None


fwer_sim = None     # fracción de repeticiones con al menos un p < 0.05 (sin efecto real)
fwer_bonf = None    # lo mismo con umbral 0.05 / m

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_l = lotes.assign(t=lotes["aprobados"] / lotes["lotes"]).set_index(["coop", "dificultad"])
assert tasas is not None and list(tasas.index) == ["A", "B"] and list(tasas.columns) == ["facil", "dificil", "global"], "tasas: índice A y B; columnas facil, dificil y global."
assert abs(tasas.loc["A", "facil"] - _l.loc[("A", "facil"), "t"]) < 1e-12 and abs(tasas.loc["B", "dificil"] - _l.loc[("B", "dificil"), "t"]) < 1e-12, "Las tasas por categoría son aprobados / lotes de esa fila."
assert abs(tasas.loc["A", "global"] - 190 / 300) < 1e-12 and abs(tasas.loc["B", "global"] - 258 / 300) < 1e-12, "La tasa global suma primero los aprobados y los lotes de las dos categorías y luego divide (no promedies las dos tasas)."
assert simpson is True, "simpson debe ser True: A gana en cada categoría pero B gana en el total."
_w = np.array([300, 300]) / 600
assert tasa_est is not None and abs(tasa_est["A"] - (_w[0] * 28 / 30 + _w[1] * 162 / 270)) < 1e-12 and abs(tasa_est["B"] - (_w[0] * 243 / 270 + _w[1] * 15 / 30)) < 1e-12, "tasa_est: cada tasa por categoría ponderada con la proporción de lotes fáciles (300/600) y difíciles (300/600) de ambas cooperativas juntas."
assert tasa_est["A"] > tasa_est["B"], "Con los mismos pesos para las dos cooperativas, A debe quedar por encima de B (la paradoja se resuelve)."
assert abs(fwer_teorico(20, 0.05) - (1 - 0.95 ** 20)) < 1e-12 and abs(fwer_teorico(1, 0.05) - 0.05) < 1e-12, "fwer_teorico = 1 - (1 - alfa) ** m."
_se = math.sqrt(0.64 * 0.36 / R4)
assert fwer_sim is not None and abs(fwer_sim - (1 - 0.95 ** 20)) < 5 * _se, "fwer_sim debería rondar 0.64: cuenta las repeticiones con ALGÚN p < 0.05 (any sobre el eje de las 20 pruebas)."
assert fwer_bonf is not None and 0.025 < fwer_bonf < 0.075 and fwer_bonf < fwer_sim / 4, "Con umbral 0.05 / m la tasa de al menos un falso positivo debe volver a ≈ 0.05."
print(f"✅ Reto 4 superado: tasas globales A {tasas.loc['A','global']:.3f} vs B {tasas.loc['B','global']:.3f} | estandarizadas A {tasa_est['A']:.3f} vs B {tasa_est['B']:.3f} | FWER teórico {fwer_teorico(20, 0.05):.3f}, simulado {fwer_sim:.3f}, Bonferroni {fwer_bonf:.3f}")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Por qué la asignación de Neyman gana al diseño proporcional y al MAS? Usa `deff`. ¿Qué información necesitas **antes** de muestrear para poder usarla y qué pasa si la desviación de un estrato está mal estimada?
- Con `tam`, ¿qué cambió y qué no al multiplicar `n` por 16? ¿Qué **supuesto** necesitan `ipw` y `post` para corregir el sesgo y qué no corrigen aunque el supuesto falle?
- ¿Por qué `pend_ingenua` mezcla dos efectos? ¿Qué te dan `pend_ajustada` y `pend_experimento`, y por qué el experimento es más convincente que el ajuste?
- En la paradoja de Simpson, ¿con cuál cooperativa te quedarías para lotes difíciles y por qué las tasas globales engañan? ¿Qué decisión de estandarización (pesos) puede cambiar el veredicto?
- ¿Qué te dice `fwer_sim` sobre mirar 20 métricas «a ver cuál sale»? Propón una defensa y su costo.

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: escalas, asignaciones, varianza teórica y simulación de tres diseños de muestreo | 20 |
| Reto 2: sesgo, varianza y ECM de la no respuesta; ponderación; más datos no corrigen el sesgo | 25 |
| Reto 3: Pearson, Spearman, correlación parcial, pendiente ajustada y experimento | 20 |
| Reto 4: paradoja de Simpson, estandarización y comparaciones múltiples (FWER) | 20 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Mis asignaciones de muestra son enteros y los tres diseños usan el mismo `n`.
- [ ] Distingo en mis palabras el sesgo (no baja con `n`) de la varianza (sí baja con `n`).
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Estadística, Análisis y Representación de Datos</i>
  </p>
</div>